# MRI: T1 and T2 relaxation and image contrast

*medical · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/mri_relaxation)

After a radio-frequency pulse, the magnetisation of each tissue grows back along the field with time constant $T_1$ and its transverse part decays with $T_2$. Evaluating the spin-echo signal as one tensor over (tissue, $TR$, $TE$) and contracting it with a tissue-label map of a cartoon brain shows how the choice of $TR$ and $TE$ makes an image T1-, PD- or T2-weighted.

**Vector form**

$$M_z(t) = M_0\,(1 - e^{-t/T_1}),\quad M_{xy}(t) = M_0\,e^{-t/T_2},\quad S \propto PD\,(1 - e^{-TR/T_1})\,e^{-TE/T_2}$$

**Index (tensor) form**

$$S_{sab} = PD_s\,\bigl(1 - e^{-TR_a/T_{1,s}}\bigr)\,e^{-TE_b/T_{2,s}},\qquad I_{xyab} = L_{xys}\,S_{sab},\qquad TI_{0,s} = T_{1,s}\ln 2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_{1,s},\ T_{2,s},\ PD_s$ | `tissue_parameters()` | (S,) | relaxation times and proton density of the $S = 5$ tissues (approximate values at 1.5 T) |
| $M_{z,st}$ | `longitudinal(t, T1, M0)` | (S, T) | longitudinal recovery, broadcast over tissue $s$ and time $t$ |
| $M_{xy,st}$ | `transverse(t, T2, M0)` | (S, T) | transverse decay of each tissue |
| $S_{sab}$ | `einsum('s,sa,sb->sab', PD, recovery, decay)` | (S, A, B) | spin-echo signal for every tissue, repetition time $TR_a$ and echo time $TE_b$ (no index summed) |
| $L_{xys}$ | `phantom(n)` | (N, N, S) | one-hot label map: which tissue fills pixel $(x, y)$ |
| $I_{xyab}$ | `einsum('xys,s...->xy...', L, S)` | (N, N, ...) | repeated $s$ ⇒ summed: the image for each $(TR, TE)$ |
| $TI_{0,s}$ | `null_point(T1)` | (S,) | inversion time where $M_0(1 - 2e^{-t/T_1})$ crosses zero, used to null fat (STIR) or CSF (FLAIR) |

Short $TR$ and short $TE$ give T1 weighting: fat and white matter are bright, CSF is dark, because its long $T_1$ has had no time to recover. Long $TR$ with short $TE$ removes the $T_1$ factor, leaving proton density (PD) weighting. Long $TR$ and long $TE$ give T2 weighting: CSF, with its very long $T_2$, becomes the brightest tissue. The simple formula ignores the refocusing pulse's effect on $M_z$ and assumes $TR \gg TE$. Each image is windowed to its own brightest pixel, as a scanner display does. The tissue values are rough textbook numbers at 1.5 T, for teaching only, not for clinical use. Try moving the short and long $TR$/$TE$ and watch the grey/white matter contrast flip.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/mri_relaxation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import null_point


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T1, T2, PD = tensors.tissue_parameters(dev)
    t_long = tensors.time_axis(5000.0, 400, dev)  # ms, for T1 recovery
    t_short = tensors.time_axis(300.0, 300, dev)  # ms, for T2 decay
    TR, TE = tensors.protocol_grids(dev)
    labels = tensors.phantom(params["size"], dev)

    protocols = {  # weighting → (TR, TE) in ms
        "T1-weighted": (params["tr_short"], params["te_short"]),
        "PD-weighted": (params["tr_long"], params["te_short"]),
        "T2-weighted": (params["tr_long"], params["te_long"]),
    }
    path = solve.sweep(TR, TE, list(protocols.values()), params["n_frames"])
    Mz, Mxy = solve.relaxation_curves(T1, T2, PD, t_long, t_short)
    S, frames = solve.solve(T1, T2, PD, TR, TE, labels, path)

    print("tissue         T1 (ms)  T2 (ms)   PD   null TI = T1 ln2 (ms)")
    for name, a, b, p, ti in zip(tensors.TISSUES, T1.tolist(), T2.tolist(), PD.tolist(), null_point(T1).tolist()):
        print(f"{name:13s} {a:8.0f} {b:8.0f} {p:5.2f} {ti:10.0f}")
    for weighting, (tr, te) in protocols.items():
        s = S[:, solve.nearest(TR, tr), solve.nearest(TE, te)]
        ranking = " > ".join(tensors.TISSUES[i] for i in s.argsort(descending=True).tolist())
        print(f"{weighting} (TR {tr:.0f}, TE {te:.0f} ms): {ranking}")
    return plot.animate(t_long, t_short, Mz, Mxy, TR, TE, path, frames, out_dir / "mri_relaxation.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/mri_relaxation/equation.py
r"""Physics → tensor translation.

After an excitation the magnetisation of every tissue s relaxes back to equilibrium M_0:

    M_{z,st}  = M_{0,s} (1 - e^{-t_t / T_{1,s}})          longitudinal recovery after a 90° pulse
    M_{z,st}  = M_{0,s} (1 - 2 e^{-t_t / T_{1,s}})        inversion recovery after a 180° pulse
    M_{xy,st} = M_{0,s} e^{-t_t / T_{2,s}}                 transverse decay

Index s labels the tissue and t the time sample; nothing is summed, the formulas broadcast over (s, t).
A spin-echo sequence repeated every TR and read out at TE gives the signal tensor over (tissue, TR, TE):

    S_{sab} = PD_s (1 - e^{-TR_a / T_{1,s}}) e^{-TE_b / T_{2,s}}

An image is the label map L_{xys} (one-hot: which tissue fills pixel x, y) contracted with S over s:

    I_{xyab} = L_{xys} S_{sab}          (s summed)
"""

import math

import torch


def longitudinal(t: torch.Tensor, T1: torch.Tensor, M0: torch.Tensor) -> torch.Tensor:
    r"""M_{z,st} = M_{0,s}(1 - e^{-t_t/T_{1,s}}).   t: (T,), T1, M0: (S,) → (S, T)"""
    return M0[:, None] * (1 - torch.exp(-t[None, :] / T1[:, None]))


def inversion_recovery(t: torch.Tensor, T1: torch.Tensor, M0: torch.Tensor) -> torch.Tensor:
    r"""M_{z,st} = M_{0,s}(1 - 2e^{-t_t/T_{1,s}}): starts at -M_0, crosses zero at T_1 \ln 2.   → (S, T)"""
    return M0[:, None] * (1 - 2 * torch.exp(-t[None, :] / T1[:, None]))


def transverse(t: torch.Tensor, T2: torch.Tensor, M0: torch.Tensor) -> torch.Tensor:
    r"""M_{xy,st} = M_{0,s} e^{-t_t/T_{2,s}}.   t: (T,), T2, M0: (S,) → (S, T)"""
    return M0[:, None] * torch.exp(-t[None, :] / T2[:, None])


def null_point(T1: torch.Tensor) -> torch.Tensor:
    r"""Inversion time where M_z = 0:  TI_s = T_{1,s} \ln 2.   (S,) → (S,)"""
    return T1 * math.log(2.0)


def spin_echo_signal(
    PD: torch.Tensor, T1: torch.Tensor, T2: torch.Tensor, TR: torch.Tensor, TE: torch.Tensor
) -> torch.Tensor:
    r"""S_{sab} = PD_s (1 - e^{-TR_a/T_{1,s}}) e^{-TE_b/T_{2,s}}.   (S,), (A,), (B,) → (S, A, B)

    The three factors depend on different index pairs; einsum multiplies them into one tensor (no sum).
    """
    recovery = 1 - torch.exp(-TR[None, :] / T1[:, None])  # (S, A)   T1 weighting
    decay = torch.exp(-TE[None, :] / T2[:, None])  # (S, B)   T2 weighting
    return torch.einsum("s,sa,sb->sab", PD, recovery, decay)


def image(labels: torch.Tensor, signal: torch.Tensor) -> torch.Tensor:
    r"""I_{xy...} = L_{xys} S_{s...}  (s summed).   labels: (N, N, S), signal: (S, ...) → (N, N, ...)"""
    return torch.einsum("xys,s...->xy...", labels, signal)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/mri_relaxation/tensors.py
"""Build the concrete tensors: tissue parameters, time axes, TR/TE grids and a label-map 'brain' phantom."""

import torch

# Approximate values at 1.5 T (times in ms, proton density PD relative to CSF). For teaching only.
TISSUES = ["white matter", "grey matter", "CSF", "fat", "muscle"]
_T1 = [600.0, 950.0, 4000.0, 250.0, 900.0]
_T2 = [80.0, 100.0, 2000.0, 70.0, 50.0]
_PD = [0.70, 0.80, 1.00, 0.90, 0.75]


def tissue_parameters(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """T1_s, T2_s, PD_s for the S = 5 tissues: three tensors of shape (S,)."""
    return (
        torch.tensor(_T1, device=device),
        torch.tensor(_T2, device=device),
        torch.tensor(_PD, device=device),
    )


def time_axis(t_max: float, n: int, device: torch.device) -> torch.Tensor:
    """t_t from 0 to t_max (ms): shape (T,)."""
    return torch.linspace(0.0, t_max, n, device=device)


def protocol_grids(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Repetition times TR_a (A,) from 100 to 5000 ms and echo times TE_b (B,) from 5 to 200 ms."""
    tr = torch.arange(100.0, 5001.0, 50.0, device=device)  # (A,) = (99,)
    te = torch.arange(5.0, 201.0, 1.0, device=device)  # (B,) = (196,)
    return tr, te


def phantom(n: int, device: torch.device) -> torch.Tensor:
    """One-hot label map L_{xys} of a cartoon axial brain slice: shape (n, n, S), zero outside the head.

    Layers from outside in: scalp fat, a muscle band, grey-matter cortex, white matter, two CSF ventricles.
    """
    y, x = torch.meshgrid(
        torch.linspace(-1, 1, n, device=device), torch.linspace(-1, 1, n, device=device), indexing="ij"
    )
    r = torch.sqrt((x / 0.80) ** 2 + (y / 0.95) ** 2)  # elliptical "radius" of the head
    label = torch.full((n, n), -1, dtype=torch.long, device=device)  # -1 = air
    label[r < 0.98] = 3  # fat (scalp)
    label[r < 0.90] = 4  # muscle / tissue under the scalp
    label[r < 0.84] = 1  # grey matter
    angle = torch.atan2(y, x)
    label[r < 0.62 + 0.08 * torch.cos(9 * angle)] = 0  # white matter with a wiggly (gyral) boundary
    for cx in (-0.17, 0.17):  # two lateral ventricles filled with CSF
        ventricle = ((x - cx) / 0.10) ** 2 + ((y + 0.05) / 0.32) ** 2 < 1
        label[ventricle] = 2
    label[(x / 0.05) ** 2 + ((y - 0.45) / 0.18) ** 2 < 1] = 2  # a CSF-filled fissure
    onehot = torch.zeros(n, n, len(TISSUES), device=device)
    inside = label >= 0
    onehot[inside, label[inside]] = 1.0
    return onehot


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/mri_relaxation/solve.py
"""Evaluate the relaxation curves and the spin-echo signal tensor, and plan a sweep through (TR, TE)."""

import torch

from .equation import image, longitudinal, spin_echo_signal, transverse


def relaxation_curves(
    T1: torch.Tensor, T2: torch.Tensor, PD: torch.Tensor, t_long: torch.Tensor, t_short: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """M_z on the long time axis (S, T) and M_xy on the short one (S, T') with M_0 = PD."""
    return longitudinal(t_long, T1, PD), transverse(t_short, T2, PD)


def nearest(grid: torch.Tensor, value: float) -> int:
    """Index of the grid point closest to value."""
    return int(torch.argmin((grid - value).abs()))


def sweep(TR: torch.Tensor, TE: torch.Tensor, corners: list[tuple[float, float]], n_frames: int) -> torch.Tensor:
    """Grid indices (F, 2) of a path that visits the (TR, TE) corners in order, moving in straight lines.

    Each corner is held for a few frames so the reader can see that weighting.
    """
    per_leg = max(2, n_frames // (2 * len(corners)))
    hold = per_leg // 2
    path = []
    for (tr0, te0), (tr1, te1) in zip(corners, corners[1:] + corners[:1]):
        path += [(tr0, te0)] * hold
        for k in range(per_leg):
            f = k / per_leg
            path.append((tr0 + f * (tr1 - tr0), te0 + f * (te1 - te0)))
    return torch.tensor([(nearest(TR, a), nearest(TE, b)) for a, b in path])


def solve(
    T1: torch.Tensor,
    T2: torch.Tensor,
    PD: torch.Tensor,
    TR: torch.Tensor,
    TE: torch.Tensor,
    labels: torch.Tensor,
    path: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the signal tensor S (S, A, B) and the phantom images along the path (F, N, N)."""
    S = spin_echo_signal(PD, T1, T2, TR, TE)  # (S, A, B)
    S_path = S[:, path[:, 0], path[:, 1]]  # (S, F)   pick the (TR, TE) of every frame
    frames = image(labels, S_path).permute(2, 0, 1)  # (N, N, F) → (F, N, N)
    return S, frames


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/mri_relaxation/plot.py
"""Animate the relaxation curves with moving TR/TE markers next to the phantom image they produce."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .tensors import TISSUES


def animate(
    t_long: torch.Tensor,
    t_short: torch.Tensor,
    Mz: torch.Tensor,
    Mxy: torch.Tensor,
    TR: torch.Tensor,
    TE: torch.Tensor,
    path: torch.Tensor,
    frames: torch.Tensor,
    out: Path,
) -> Path:
    tl, ts = t_long.cpu().numpy(), t_short.cpu().numpy()
    Mz, Mxy = Mz.cpu().numpy(), Mxy.cpu().numpy()  # (S, T), (S, T')
    tr = TR.cpu().numpy()[path[:, 0].numpy()]  # (F,) TR of each frame
    te = TE.cpu().numpy()[path[:, 1].numpy()]  # (F,) TE of each frame
    images = frames.cpu().numpy()  # (F, N, N)

    fig = plt.figure(figsize=(7.4, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.25, 1])
    ax_z = fig.add_subplot(grid[0, 0])
    ax_xy = fig.add_subplot(grid[1, 0])
    ax_im = fig.add_subplot(grid[:, 1])

    for s, name in enumerate(TISSUES):
        ax_z.plot(tl, Mz[s], color=ACCENT[s], lw=1.5, label=name)
        ax_xy.plot(ts, Mxy[s], color=ACCENT[s], lw=1.5)
    ax_z.set_xlim(0, tl[-1])
    ax_z.set_ylim(0, 1.05)
    ax_z.set_ylabel("$M_z$ (recovery)")
    ax_z.set_xlabel("t (ms)", labelpad=0)
    ax_z.legend(fontsize=6.5, loc="lower right", ncol=2)
    ax_xy.set_xlim(0, ts[-1])
    ax_xy.set_ylim(0, 1.05)
    ax_xy.set_ylabel("$M_{xy}$ (decay)")
    ax_xy.set_xlabel("t (ms)", labelpad=0)
    tr_line = ax_z.axvline(tr[0], color="#111827", ls="--", lw=1)
    te_line = ax_xy.axvline(te[0], color="#111827", ls="--", lw=1)
    tr_dots = ax_z.scatter(np.zeros(len(TISSUES)), np.zeros(len(TISSUES)), c=ACCENT[: len(TISSUES)], s=18, zorder=3)
    te_dots = ax_xy.scatter(np.zeros(len(TISSUES)), np.zeros(len(TISSUES)), c=ACCENT[: len(TISSUES)], s=18, zorder=3)

    shown = ax_im.imshow(images[0], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax_im.set_xticks([])
    ax_im.set_yticks([])
    ax_im.grid(False)
    title = ax_im.set_title("")

    def weighting(a: float, b: float) -> str:
        if a < 1500 and b < 40:
            return "T1-weighted"
        if a >= 1500 and b < 40:
            return "PD-weighted"
        if a >= 1500 and b >= 60:
            return "T2-weighted"
        return "mixed"

    def update(step: int):
        a, b = tr[step], te[step]
        tr_line.set_xdata([a, a])
        te_line.set_xdata([b, b])
        tr_dots.set_offsets(np.c_[np.full(len(TISSUES), a), Mz[:, np.abs(tl - a).argmin()]])
        te_dots.set_offsets(np.c_[np.full(len(TISSUES), b), Mxy[:, np.abs(ts - b).argmin()]])
        img = images[step]
        shown.set_data(img / max(img.max(), 1e-9))  # window each image to its brightest pixel
        title.set_text(f"{weighting(a, b)}\nTR = {a:.0f} ms, TE = {b:.0f} ms")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(tr), len(tr)), out, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'tr_short': 500.0,
    'tr_long': 4500.0,
    'te_short': 15.0,
    'te_long': 100.0,
    'size': 96,
    'n_frames': 90
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.mri_relaxation")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.mri_relaxation.main")
problem_dir = Path("problems/medical/mri_relaxation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
